# Coarse-Graining: GPCCA · Escape · MFPT · Validation

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, SpectralClustering
from scipy.linalg import inv
from pygpcca import GPCCA
import warnings; warnings.filterwarnings('ignore')

# ── key parameter ─────────────────────────────────────────────────────────────
N_CLUSTERS = 4

# ── data paths ────────────────────────────────────────────────────────────────
A_samples  = np.load('data/A_England_Delta.npy')   # (100, 50, 50)
Ne_samples = np.load('data/Ne_England_Delta.npy')  # (100, 50)

n_samples, n_demes, _ = A_samples.shape
A  = A_samples.mean(0); A = A / A.sum(1, keepdims=True)
Ne = Ne_samples.mean(0)

print(f"n_demes={n_demes}, N_CLUSTERS={N_CLUSTERS}")
print(f"Ne mean={Ne.mean():.0f}, A_ii mean={np.diag(A).mean():.3f}")


In [ ]:
# shared helpers

def stationary(A):
    n = A.shape[0]; M = A.T - np.eye(n); M[-1] = 1
    b = np.zeros(n); b[-1] = 1
    return np.linalg.solve(M, b)

def timescales(A):
    ev = np.sort(np.abs(np.linalg.eigvals(A)))[::-1]
    return -1 / np.log(np.clip(ev[1:], 1e-12, 1-1e-12))

def coarse_A(labels, A, Ne, m):
    P = np.zeros((m, m))
    for k in range(m):
        dk = np.where(labels==k)[0]
        if not len(dk): P[k,k]=1; continue
        den = sum(A[i,j]*Ne[j] for i in dk for j in range(len(Ne)))
        for l in range(m):
            dl = np.where(labels==l)[0]
            P[k,l] = sum(A[i,j]*Ne[j] for i in dk for j in dl)/den if den else 0
    P = np.clip(P,0,None); return P / P.sum(1, keepdims=True)

def int_flow(labels, A, m):
    out = []
    for k in range(m):
        d = np.where(labels==k)[0]
        out.append(float(A[np.ix_(d,d)].sum(1).mean()) if len(d)>1 else 0.)
    return out

def wf_step(p, A, Ne):
    pm = A @ p; pn = np.zeros_like(p)
    for i in range(len(p)):
        n = max(int(round(Ne[i])),1)
        pn[i] = np.random.binomial(n, float(np.clip(pm[i],0,1))) / n
    return pn

def wf_traj(p0, A, Ne, T):
    p = p0.copy(); out = [p.copy()]
    for _ in range(T): p = wf_step(p,A,Ne); out.append(p.copy())
    return np.array(out)

def proj_fuzzy(traj, chi, Ne, m):
    out = np.zeros((len(traj), m))
    for k in range(m):
        w = chi[:,k]*Ne
        if w.sum()>0: out[:,k] = traj @ w / w.sum()
    return out

def proj_hard(traj, labels, Ne, m):
    out = np.zeros((len(traj), m))
    for k in range(m):
        d = np.where(labels==k)[0]
        if not len(d): continue
        w = Ne[d]/Ne[d].sum(); out[:,k] = traj[:,d] @ w
    return out

print("helpers ready")


## 1. GPCCA

In [ ]:
g = GPCCA(A)
g.optimize(m=N_CLUSTERS)

chi   = g.memberships
P_g   = np.clip(g.coarse_grained_transition_matrix, 0, None)
P_g   = P_g / P_g.sum(1, keepdims=True)
Ne_g  = chi.T @ Ne
lbl_g = np.argmax(chi, 1)

print("Coarse A (GPCCA):"); print(np.round(P_g, 4))
print("\nCluster sizes:", [(lbl_g==k).sum() for k in range(N_CLUSTERS)])
print("Internal flows:", [round(f,3) for f in int_flow(lbl_g, A, N_CLUSTERS)])
print("\nTimescales:")
ts_f = timescales(A); ts_g = timescales(P_g)
for k in range(min(N_CLUSTERS-1, 4)):
    print(f"  tau_{k+2}: full={ts_f[k]:.2f}  GPCCA={ts_g[k]:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.imshow(chi.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_xlabel('Deme'); ax.set_ylabel('Super-deme')
ax.set_yticks(range(N_CLUSTERS))
ax.set_yticklabels([f'S{k+1}' for k in range(N_CLUSTERS)])
ax.set_title('GPCCA membership chi')

ax = axes[1]
sns.heatmap(P_g, ax=ax, annot=True, fmt='.3f', cmap='Blues',
            square=True, cbar=False,
            xticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)],
            yticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)])
ax.set_title('Coarse transition matrix')
plt.tight_layout(); plt.show()


## 2. Escape Hitting Distribution

In [ ]:
NE_SCALE   = Ne.mean() / 5.0
N_TRIALS   = 500
THRESHOLD  = 0.5

Ne_sc = np.maximum(Ne / NE_SCALE, 1.0)

def first_hit(origin, A, Ne, thr=0.5, max_gen=2000):
    p = np.zeros(len(Ne)); p[origin] = 1./max(Ne[origin],1.)
    for _ in range(max_gen):
        p = wf_step(p, A, Ne)
        hits = np.where(p > thr)[0]
        if len(hits): return int(hits[np.argmax(p[hits])])
        if p.sum() < 1e-9: return -1
    return -1

H = np.zeros((n_demes, n_demes))
for origin in range(n_demes):
    counts = np.zeros(n_demes); lost = 0
    for _ in range(N_TRIALS):
        d = first_hit(origin, A, Ne_sc, THRESHOLD)
        if d >= 0: counts[d] += 1
        else: lost += 1
    tot = counts.sum()
    H[origin] = counts/tot if tot>0 else 0.
    if (origin+1) % 10 == 0:
        print(f"  deme {origin+1}/50  loss={lost/N_TRIALS:.2f}")
print("done")


In [ ]:
# zero diagonal, renormalise, cluster
H_esc = H.copy(); np.fill_diagonal(H_esc, 0.)
rs = H_esc.sum(1, keepdims=True); rs[rs<1e-9]=1.
H_esc /= rs

km = KMeans(n_clusters=N_CLUSTERS, n_init=50, random_state=42).fit(H_esc)
lbl_e = km.labels_
# reorder by internal flow
order = np.argsort(int_flow(lbl_e, A, N_CLUSTERS))
remap = {o:n for n,o in enumerate(order)}
lbl_e = np.array([remap[l] for l in lbl_e])

P_e   = coarse_A(lbl_e, A, Ne, N_CLUSTERS)
Ne_e  = np.array([(Ne[lbl_e==k].sum() if (lbl_e==k).sum()>0 else 1.)
                   for k in range(N_CLUSTERS)])

print("Cluster sizes:", [(lbl_e==k).sum() for k in range(N_CLUSTERS)])
print("Internal flows:", [round(f,3) for f in int_flow(lbl_e, A, N_CLUSTERS)])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
oi = np.argsort(lbl_e)
ax.imshow(H_esc[np.ix_(oi,oi)], aspect='auto', cmap='Blues')
ax.set_title('Escape distribution (sorted by cluster)')
ax.set_xlabel('Destination'); ax.set_ylabel('Origin')
bounds = np.cumsum([(lbl_e==k).sum() for k in range(N_CLUSTERS)])[:-1]
for b in bounds: ax.axhline(b-.5,color='r',lw=1); ax.axvline(b-.5,color='r',lw=1)

ax = axes[1]
sns.heatmap(P_e, ax=ax, annot=True, fmt='.3f', cmap='Blues',
            square=True, cbar=False,
            xticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)],
            yticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)])
ax.set_title('Coarse transition matrix')
plt.tight_layout(); plt.show()


## 3. MFPT

In [ ]:
pi   = stationary(A)
Pi   = np.tile(pi, (n_demes,1))
Z    = inv(np.eye(n_demes) - A + Pi)

T = np.zeros((n_demes, n_demes))
for i in range(n_demes):
    for j in range(n_demes):
        if i!=j: T[i,j] = (Z[j,j]-Z[i,j])/pi[j]
T = np.clip(T, 0, np.percentile(T[T>0], 95))

tau = np.median(T[T>0])
S   = np.exp(-(T+T.T)/(2*tau)); np.fill_diagonal(S,0.)

sc = SpectralClustering(n_clusters=N_CLUSTERS, affinity='precomputed',
                        n_init=50, random_state=42).fit(S)
lbl_m = sc.labels_
order = np.argsort(int_flow(lbl_m, A, N_CLUSTERS))
remap = {o:n for n,o in enumerate(order)}
lbl_m = np.array([remap[l] for l in lbl_m])

P_m   = coarse_A(lbl_m, A, Ne, N_CLUSTERS)
Ne_m  = np.array([(Ne[lbl_m==k].sum() if (lbl_m==k).sum()>0 else 1.)
                   for k in range(N_CLUSTERS)])

print("Cluster sizes:", [(lbl_m==k).sum() for k in range(N_CLUSTERS)])
print("Internal flows:", [round(f,3) for f in int_flow(lbl_m, A, N_CLUSTERS)])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
oi = np.argsort(lbl_m)
im = ax.imshow(T[np.ix_(oi,oi)], aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='generations')
ax.set_title('MFPT matrix (sorted by cluster)')
ax.set_xlabel('Destination'); ax.set_ylabel('Origin')
bounds = np.cumsum([(lbl_m==k).sum() for k in range(N_CLUSTERS)])[:-1]
for b in bounds: ax.axhline(b-.5,color='b',lw=1); ax.axvline(b-.5,color='b',lw=1)

ax = axes[1]
sns.heatmap(P_m, ax=ax, annot=True, fmt='.3f', cmap='Blues',
            square=True, cbar=False,
            xticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)],
            yticklabels=[f'S{k+1}' for k in range(N_CLUSTERS)])
ax.set_title('Coarse transition matrix')
plt.tight_layout(); plt.show()


## 4. Validation

In [ ]:
N_STEPS = 200
N_SIMS  = 60

methods = {
    'GPCCA':  (lbl_g, Ne_g,  P_g, 'fuzzy', chi),
    'Escape': (lbl_e, Ne_e,  P_e, 'hard',  None),
    'MFPT':   (lbl_m, Ne_m,  P_m, 'hard',  None),
}

results = {}
for ic, p0_fn in [
    ('uniform',   lambda: np.full(n_demes, 0.1)),
    ('localised', lambda: np.concatenate([[0.5], np.zeros(n_demes-1)])),
]:
    results[ic] = {}
    for name, (lbl, Ne_c, P_c, ptype, ch) in methods.items():
        tp, tc = [], []
        p0_full = p0_fn()
        if ptype=='fuzzy':
            p0c = np.array([np.dot(ch[:,k]*Ne, p0_full)/max((ch[:,k]*Ne).sum(),1e-9)
                            for k in range(N_CLUSTERS)])
        else:
            p0c = np.array([(Ne[lbl==k]*p0_full[lbl==k]).sum()/max(Ne[lbl==k].sum(),1e-9)
                            if (lbl==k).sum()>0 else 0.1 for k in range(N_CLUSTERS)])
        for _ in range(N_SIMS):
            tf = wf_traj(p0_fn(), A, Ne, N_STEPS)
            tp.append(proj_fuzzy(tf,ch,Ne,N_CLUSTERS) if ptype=='fuzzy'
                      else proj_hard(tf,lbl,Ne,N_CLUSTERS))
            tc.append(wf_traj(p0c.copy(), P_c, Ne_c, N_STEPS))
        results[ic][name] = {'proj': np.array(tp), 'coarse': np.array(tc)}
    print(f"done: {ic}")


In [ ]:
# variance ratio table
t_check = [20, 100, N_STEPS]
for ic in ['uniform','localised']:
    print(f"\n{ic.upper()}")
    print(f"{'':>10}" + "".join(f"  t={t:>4}" for t in t_check))
    for name in methods:
        d = results[ic][name]; p = d['proj']; c = d['coarse']
        row = f"{name:>10}"
        for t in t_check:
            vr = np.nanmean([c[:,t,k].var()/p[:,t,k].var()
                             if p[:,t,k].var()>1e-10 else np.nan
                             for k in range(N_CLUSTERS)])
            row += f"  {vr:>6.3f}"
        print(row)
print("\nideal = 1.0")


In [ ]:
# variance ratio over time
t_arr = np.arange(N_STEPS+1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ls = {'GPCCA':'-','Escape':'--','MFPT':':'}

for col, ic in enumerate(['uniform','localised']):
    ax = axes[col]
    for name in methods:
        d = results[ic][name]; p = d['proj']; c = d['coarse']
        vr = [np.nanmean([c[:,t,k].var()/p[:,t,k].var()
                          if p[:,t,k].var()>1e-10 else np.nan
                          for k in range(N_CLUSTERS)])
              for t in range(N_STEPS+1)]
        ax.plot(t_arr, vr, lw=2, ls=ls[name], label=name)
    ax.axhline(1., color='k', lw=1, ls='--')
    ax.set_xlabel('Generation'); ax.set_ylabel('Variance ratio')
    ax.set_title(f'{ic} IC'); ax.set_ylim(0, 2.2)
    ax.legend()

plt.suptitle(f'Variance ratio (coarse/full) — N_CLUSTERS={N_CLUSTERS}')
plt.tight_layout(); plt.show()


In [ ]:
# per-cluster trajectories: mean ± 1 std, localised IC
fig, axes = plt.subplots(2, N_CLUSTERS, figsize=(4*N_CLUSTERS, 7))
if N_CLUSTERS == 1: axes = axes.reshape(2,1)

for k in range(N_CLUSTERS):
    for row, ic in enumerate(['uniform','localised']):
        ax = axes[row, k]
        for name in methods:
            d = results[ic][name]
            mf = d['proj'][:,:,k].mean(0);   sf = d['proj'][:,:,k].std(0)
            mc = d['coarse'][:,:,k].mean(0)
            ax.fill_between(t_arr, mf-sf, mf+sf, alpha=0.15)
            ax.plot(t_arr, mf, lw=1.5, alpha=0.6, label=f'{name} full')
            ax.plot(t_arr, mc, lw=1.5, ls=ls[name], label=f'{name} coarse')
        ax.set_title(f'S{k+1} — {ic}')
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel('Generation')
        if k==0: ax.set_ylabel('p'); ax.legend(fontsize=7)

plt.tight_layout(); plt.show()


## 5. Summary Comparison

In [ ]:
from itertools import permutations

def best_agreement(l1, l2, m):
    return max((l1==np.array([p[x] for x in l2])).mean()
               for p in permutations(range(m)))

# timescales
print("Timescales:")
print(f"{'':>8}" + f"{'full':>8}" +
      "".join(f"{n:>10}" for n in methods))
ts_all = {n: timescales(P) for n,(lbl,Ne_c,P,*_) in methods.items()}
for k in range(min(N_CLUSTERS-1,5)):
    row = f"  tau_{k+2}  {timescales(A)[k]:>7.2f}"
    for n in methods: row += f"  {ts_all[n][k]:>8.2f}"
    print(row)

# internal flow
print("\nInternal flow:")
for n,(lbl,*_) in methods.items():
    f = int_flow(lbl,A,N_CLUSTERS)
    print(f"  {n:>10}: {[round(x,3) for x in f]}  mean={np.mean(f):.3f}")

# cluster agreement
print("\nCluster agreement:")
lbls = {n:lbl for n,(lbl,*_) in methods.items()}
pairs = [('GPCCA','Escape'),('GPCCA','MFPT'),('Escape','MFPT')]
for m1,m2 in pairs:
    print(f"  {m1} vs {m2}: {best_agreement(lbls[m1],lbls[m2],N_CLUSTERS):.1%}")
